# Workflow Complet : ResNet50V2 sur GTEx 11 Classes (Expérience A)

**Protocole d'Entraînement :**
- Backbone : `ResNet50V2` (poids ImageNet)
- Input : `224x224x3` (RGB, float32, `[0,255]`)
- Classes : 11 (Bladder, Brain, Cerebellum, Kidney, Liver, Lung, Muscle, Oesophagus, Pancreas, Spleen, Testis)
- Stratégie : 3 phases (Tête, 30% Fine-Tuning, Full Fine-Tuning)
- Précision Mixte : `mixed_float16`
- Environnement cible : Kaggle


## ⚠️ Avertissement Légal et Médical
Ce modèle est développé exclusivement à des fins de **recherche éducative et de démonstration technique**. 
Il ne s'agit pas d'un dispositif médical certifié. Les prédictions générées par ce réseau neuronal ne doivent en **aucun cas** être utilisées pour le diagnostic clinique, la prise de décision thérapeutique, ou remplacer le jugement d'un médecin anatomopathologiste qualifié.


In [ ]:
REPO_URL = "https://github.com/MyElhadri/histology-ai-classification.git"
BRANCH = "main"

PROJECT_DIR = "/kaggle/working/histology-ai-classification"
OUTPUT_DIR = "/kaggle/working/resnet50v2-gtex-11-exp-a"

CONFIG_PATH = (
    "configs/experiments/"
    "resnet50v2_gtex_11_exp_a.yaml"
)

# ⚠️ ACTIVER RUN_TRAINING=True SEULEMENT APRÈS RÉUSSITE DE :
# - détection du dataset
# - audit
# - tests
# - dry-run
# - smoke test éventuel.

RUN_TESTS = True
RUN_DATASET_AUDIT = True
RUN_DRY_RUN = True

RUN_SMOKE_TEST = False
RUN_TRAINING = False
RUN_VALIDATION_EVALUATION = False
RUN_FINAL_TEST = False

GENERATE_REPORT = False
RESUME_TRAINING = True
ALLOW_OVERWRITE = False


In [ ]:
# === Validation de la Configuration ===
# Détecte les combinaisons de flags incohérentes AVANT toute exécution.

if RUN_SMOKE_TEST and RUN_TRAINING:
    raise ValueError(
        "INTERDIT : RUN_SMOKE_TEST et RUN_TRAINING ne peuvent pas être True simultanément."
    )

if RUN_FINAL_TEST and RUN_SMOKE_TEST:
    raise ValueError(
        "INTERDIT : Le test final ne peut pas être lancé sur un modèle smoke test."
    )

if RUN_FINAL_TEST and not RUN_TRAINING:
    raise ValueError(
        "INTERDIT : RUN_FINAL_TEST=True nécessite un entraînement complété (RUN_TRAINING=True)."
    )

if RUN_VALIDATION_EVALUATION and not RUN_TRAINING and not RUN_SMOKE_TEST:
    raise ValueError(
        "INTERDIT : RUN_VALIDATION_EVALUATION=True nécessite un modèle entraîné."
    )

if RESUME_TRAINING and ALLOW_OVERWRITE:
    raise ValueError(
        "INTERDIT : RESUME_TRAINING et ALLOW_OVERWRITE ne peuvent pas être True simultanément."
    )

print("Configuration validée :")
print(f"  RUN_TESTS             = {RUN_TESTS}")
print(f"  RUN_DATASET_AUDIT     = {RUN_DATASET_AUDIT}")
print(f"  RUN_DRY_RUN           = {RUN_DRY_RUN}")
print(f"  RUN_SMOKE_TEST        = {RUN_SMOKE_TEST}")
print(f"  RUN_TRAINING          = {RUN_TRAINING}")
print(f"  RUN_VALIDATION_EVALUATION = {RUN_VALIDATION_EVALUATION}")
print(f"  RUN_FINAL_TEST        = {RUN_FINAL_TEST}")
print(f"  GENERATE_REPORT       = {GENERATE_REPORT}")
print(f"  RESUME_TRAINING       = {RESUME_TRAINING}")
print(f"  ALLOW_OVERWRITE       = {ALLOW_OVERWRITE}")


In [ ]:
import os

in_kaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''
if not in_kaggle:
    print("⚠️ AVERTISSEMENT : Environnement Kaggle NON détecté. Ce notebook est optimisé pour Kaggle.")
else:
    print("Environnement Kaggle détecté avec succès.")


In [ ]:
import tensorflow as tf

print(f"Version TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        print(f"GPU détecté : {gpu}")
    if len(gpus) > 1:
        print(f"Note : {len(gpus)} GPU détectés. Le code utilise une seule GPU (pas de MirroredStrategy).")
    import subprocess
    try:
        print(subprocess.check_output(["nvidia-smi"], text=True))
    except Exception as e:
        print(f"Impossible d'exécuter nvidia-smi : {e}")
else:
    if RUN_TRAINING or RUN_SMOKE_TEST:
        raise RuntimeError(
            "AUCUN GPU DÉTECTÉ ! Entraînement et Smoke Test nécessitent un GPU. "
            "Seuls Tests, Audit et Dry-Run sont autorisés sans GPU."
        )
    print("⚠️ AUCUN GPU DÉTECTÉ. Exécution en mode CPU (Tests, Audit et Dry-Run autorisés).")

print(f"\nStratégie TensorFlow : {tf.distribute.get_strategy().__class__.__name__}")


In [ ]:
if gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"Policy mixed precision : {tf.keras.mixed_precision.global_policy().name}")
else:
    print("Mode CPU / T4 non optimal, mixed_float16 non activé.")


In [ ]:
from pathlib import Path

DATASET_DIR = None
matches = list(Path("/kaggle/input").rglob("GTEx_11_classes"))

if len(matches) == 1:
    DATASET_DIR = matches[0]
    print(f"Dataset GTEx détecté automatiquement : {DATASET_DIR}")
    
    # Vérification immédiate
    required = [
        "train", "validation", "test",
        "metadata/train.csv", "metadata/validation.csv", 
        "metadata/test.csv", "metadata/class_mapping.json"
    ]
    for req in required:
        if not (DATASET_DIR / req).exists():
            raise FileNotFoundError(f"Élément requis introuvable dans le dataset : {req}")
    print("Tous les éléments requis du dataset sont présents.")
    
elif len(matches) > 1:
    print("Plusieurs dossiers GTEx_11_classes détectés :")
    for m in matches:
        print(f" - {m}")
    raise RuntimeError("Erreur : Impossible de déterminer quel dataset utiliser.")
else:
    print("Dossiers actuels sous /kaggle/input :")
    import os
    for root, dirs, files in os.walk("/kaggle/input"):
        for d in dirs:
            print(os.path.join(root, d))
        for f in files:
            if "class_mapping.json" in f:
                print(f"Trouvé class_mapping : {os.path.join(root, f)}")
    raise RuntimeError("Erreur : GTEx_11_classes introuvable. Veuillez attacher le dataset au notebook.")


In [ ]:
import subprocess
import os

if not os.path.exists(PROJECT_DIR):
    print(f"Clonage du dépôt GitHub depuis {REPO_URL} (branche {BRANCH})...")
    subprocess.run(["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"Le dépôt existe déjà. Vérification des modifications locales...")
    status = subprocess.check_output(["git", "status", "--porcelain"], cwd=PROJECT_DIR, text=True)
    if status.strip():
        raise RuntimeError(f"Le clone contient des modifications locales non commitées :\n{status}")
    
    print("Mise à jour (git pull --ff-only)...")
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR, check=True)

# Affichage des informations Git
print("\nInformations Git :")
print("Remote :", subprocess.check_output(["git", "remote", "-v"], cwd=PROJECT_DIR, text=True).strip())
print("Branche :", subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_DIR, text=True).strip())
print("Statut :\n", subprocess.check_output(["git", "status", "--short"], cwd=PROJECT_DIR, text=True).strip())


In [ ]:
import subprocess

def get_git_commit():
    try:
        hash_full = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
        msg = subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=PROJECT_DIR, text=True).strip()
        return f"{hash_full} ({msg})"
    except subprocess.CalledProcessError as e:
        return f"Inconnu (erreur {e})"

print(f"Commit actuel : {get_git_commit()}")


In [ ]:
from pathlib import Path

required_files = [
    "src/models/resnet50v2.py",
    "src/models/heads.py",
    "src/data/gtex_pipeline.py",
    "src/data/gtex_integrity.py",
    "configs/experiments/resnet50v2_gtex_11_exp_a.yaml",
    "scripts/train_resnet50v2_gtex.py",
    "scripts/evaluate_resnet50v2_gtex.py",
    "scripts/generate_resnet50v2_gtex_report.py",
    "tests/test_resnet50v2_gtex.py",
    "tests/test_gtex_dataset_integrity.py",
    "tests/test_resnet50v2_gtex_report.py"
]

missing = False
for rel_path in required_files:
    p = Path(PROJECT_DIR) / rel_path
    if p.exists():
        print(f"OK : {rel_path}")
    else:
        print(f"ABSENT : {rel_path}")
        missing = True

if missing:
    raise RuntimeError("Erreur : Des fichiers obligatoires sont manquants.")


In [ ]:
import importlib.util
import subprocess
import sys

deps = ["pytest", "yaml", "sklearn", "pandas", "matplotlib"]
pip_names = {"yaml": "pyyaml", "sklearn": "scikit-learn"}

to_install = []
for dep in deps:
    if importlib.util.find_spec(dep) is None:
        to_install.append(pip_names.get(dep, dep))

if to_install:
    print(f"Installation des paquets manquants : {to_install}")
    subprocess.run([sys.executable, "-m", "pip", "install"] + to_install, check=True)
else:
    print("Toutes les dépendances Python sont déjà installées.")

# Affichage des versions
print("\nVersions :")
import tensorflow as tf; print(f"TensorFlow : {tf.__version__}")
import keras; print(f"Keras : {keras.__version__}")
import numpy as np; print(f"NumPy : {np.__version__}")
import pandas as pd; print(f"Pandas : {pd.__version__}")
import sklearn; print(f"Scikit-learn : {sklearn.__version__}")
import matplotlib; print(f"Matplotlib : {matplotlib.__version__}")
import yaml; print(f"PyYAML : {yaml.__version__}")


In [ ]:
from pathlib import Path

if RUN_DATASET_AUDIT:
    print(f"Racine du dataset : {DATASET_DIR}")
    for split in ["train", "validation", "test"]:
        split_dir = DATASET_DIR / split
        if split_dir.exists():
            classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
            print(f"Split {split} : {len(classes)} classes -> {classes}")
            
    print("Contenu de metadata :")
    meta = DATASET_DIR / "metadata"
    if meta.exists():
        for f in meta.iterdir():
            print(f" - {f.name}")
else:
    print("Audit d'arborescence ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_DATASET_AUDIT:
    print("Lancement de l'audit d'intégrité GTEx (comptes, class_mapping, et isolation des donneurs)...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    env["MPLBACKEND"] = "Agg"
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    report_json = f"{OUTPUT_DIR}/dataset_integrity.json"
    
    audit_cmd = [
        sys.executable, "-u", "-c", 
        "import sys; "
        "from src.data.gtex_integrity import audit_gtex_dataset; "
        "audit_gtex_dataset(sys.argv[1], sys.argv[2])",
        str(DATASET_DIR), report_json
    ]
    
    process = subprocess.Popen(
        audit_cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in process.stdout:
        print(line, end="")
        
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError(f"L'audit du dataset a échoué (Code {return_code}). Entraînement bloqué.")
        
    if not os.path.exists(report_json):
        raise RuntimeError("Le rapport dataset_integrity.json n'a pas été créé.")
        
    print("\nAudit du dataset réussi avec succès.")
else:
    print("RUN_DATASET_AUDIT = False. Ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_TESTS:
    print("Lancement de la suite de tests unitaires ciblés ResNet50V2...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["MPLBACKEND"] = "Agg"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    
    test_cmd = [
        sys.executable, "-u", "-m", "pytest",
        "tests/test_resnet50v2_gtex.py",
        "tests/test_gtex_dataset_integrity.py",
        "tests/test_resnet50v2_gtex_report.py",
        "-vv", "-s", "--tb=long", "--maxfail=1"
    ]
    
    process = subprocess.Popen(
        test_cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    failed_test_info = None
    
    for line in process.stdout:
        print(line, end="")
        if line.startswith("FAILED tests/"):
            failed_test_info = line.strip()
            
    return_code = process.wait()
    
    if return_code != 0:
        print("\n" + "="*70)
        print("LES TESTS ONT ÉCHOUÉ")
        print("="*70)
        if failed_test_info:
            parts = failed_test_info.split(" - ")[0].replace("FAILED ", "").split("::")
            if len(parts) >= 2:
                print(f"Fichier en échec : {parts[0]}")
                print(f"Test en échec : {parts[-1]}")
        print(f"Code de retour : {return_code}")
        print("INTERDICTION DE POURSUIVRE L'ENTRAÎNEMENT. Corrigez le code source d'abord.")
        print("="*70 + "\n")
        raise RuntimeError(f"Pytest a échoué avec le code {return_code}.")
        
    print("\nTous les tests unitaires ont réussi avec succès.")
else:
    print("RUN_TESTS = False. Étape de test ignorée.")


In [ ]:
import subprocess
import os
import sys

if RUN_DRY_RUN:
    print("Exécution du Dry-Run pour valider la configuration...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    env["MPLBACKEND"] = "Agg"
    
    dry_run_dir = f"{OUTPUT_DIR}/preflight/dry_run"
    os.makedirs(dry_run_dir, exist_ok=True)
    
    cmd = [
        sys.executable, "-u", "scripts/train_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", dry_run_dir,
        "--dry-run"
    ]
    
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in process.stdout:
        print(line, end="")
        
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("Le Dry-Run a échoué.")
    print("\nDry-Run validé.")
else:
    print("RUN_DRY_RUN = False. Ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_SMOKE_TEST and RUN_TRAINING:
    raise ValueError("INTERDIT: RUN_SMOKE_TEST et RUN_TRAINING ne peuvent pas être True en même temps.")

if RUN_SMOKE_TEST or RUN_TRAINING:
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    env["MPLBACKEND"] = "Agg"
    
    cmd = [
        sys.executable, "-u", "scripts/train_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--dataset-dir", str(DATASET_DIR)
    ]
    
    # Le script train_resnet50v2_gtex.py ajoute automatiquement /smoke_test
    # au output-dir quand --smoke-test est passé. Ne PAS doubler le suffixe.
    if RUN_SMOKE_TEST:
        print("Démarrage du SMOKE TEST (NON SCIENTIFIQUE)...\n")
        SMOKE_OUTPUT_DIR = f"{OUTPUT_DIR}-smoke-test"
        cmd.append("--smoke-test")
        cmd.extend(["--output-dir", SMOKE_OUTPUT_DIR])
        cmd.append("--overwrite")
    elif RUN_TRAINING:
        print("Démarrage de l'ENTRAÎNEMENT SCIENTIFIQUE (3 Phases)...\n")
        cmd.extend(["--output-dir", OUTPUT_DIR])
        if RESUME_TRAINING:
            cmd.append("--resume")
        if ALLOW_OVERWRITE:
            cmd.append("--overwrite")
            
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in process.stdout:
        print(line, end="")
        
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("L'entraînement a échoué.")
        
    # Validation du checkpoint
    if RUN_SMOKE_TEST:
        # Le script ajoute /smoke_test au chemin
        output_target = f"{SMOKE_OUTPUT_DIR}/smoke_test"
    else:
        output_target = OUTPUT_DIR
    best_model = f"{output_target}/checkpoints/best_model.keras"
    if not os.path.exists(best_model):
        raise RuntimeError(f"Le fichier best_model.keras n'a pas été produit à {best_model}.")
        
    print("\nProcessus terminé avec succès.")
else:
    print("Ni Smoke Test ni Entraînement demandés. Ignoré.")


In [ ]:
import subprocess
import os
import sys
import pandas as pd

if RUN_VALIDATION_EVALUATION:
    print("Démarrage de l'évaluation sur le split Validation...\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    env["MPLBACKEND"] = "Agg"
    
    if RUN_SMOKE_TEST:
        output_target = f"{SMOKE_OUTPUT_DIR}/smoke_test"
    else:
        output_target = OUTPUT_DIR
    best_model = f"{output_target}/checkpoints/best_model.keras"
    
    if not os.path.exists(best_model):
        raise FileNotFoundError(f"Impossible de trouver le modèle : {best_model}")
    
    cmd = [
        sys.executable, "-u", "scripts/evaluate_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--split", "validation",
        "--checkpoint", best_model,
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", output_target
    ]
    
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in process.stdout:
        print(line, end="")
        
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("L'évaluation de validation a échoué.")
        
    # Vérification du CSV produit
    pred_csv = os.path.join(output_target, "validation_predictions.csv")
    if not os.path.exists(pred_csv):
        raise RuntimeError("validation_predictions.csv n'a pas été généré.")
        
    df = pd.read_csv(pred_csv)
    if not RUN_SMOKE_TEST and len(df) != 8114:
        raise RuntimeError(f"Expected 8114 predictions, got {len(df)}")
        
    prob_cols = [c for c in df.columns if c.startswith("prob_")]
    if len(prob_cols) != 11:
        raise RuntimeError(f"Expected 11 probability columns, got {len(prob_cols)}")
        
    if df[prob_cols].isna().any().any():
        raise RuntimeError("NaN values found in probability columns")
        
    sums = df[prob_cols].sum(axis=1)
    import numpy as np
    if not np.allclose(sums, 1.0, atol=1e-3):
        raise RuntimeError("Probabilities do not sum to 1")
    
    print("\nÉvaluation Validation terminée avec vérifications réussies.")
    
    if GENERATE_REPORT:
        print("\nGénération des rapports visuels...")
        rep_cmd = [
            sys.executable, "-u", "scripts/generate_resnet50v2_gtex_report.py",
            "--results-dir", output_target,
            "--class-mapping", str(DATASET_DIR / "metadata" / "class_mapping.json")
        ]
        subprocess.run(rep_cmd, cwd=PROJECT_DIR, env=env, check=True)
        print("Rapports générés.")
else:
    print("RUN_VALIDATION_EVALUATION = False. Ignoré.")


In [ ]:
import subprocess
import os
import sys

if RUN_FINAL_TEST:
    if RUN_SMOKE_TEST:
        raise ValueError("INTERDIT : Impossible de lancer le TEST FINAL sur un modèle Smoke Test.")
        
    print("Démarrage de l'évaluation sur le split TEST...\n")
    print("⚠️ AVERTISSEMENT : LE TEST INDÉPENDANT NE DOIT ÊTRE ÉVALUÉ QU'UNE SEULE FOIS.\n")
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    env["MPLBACKEND"] = "Agg"
    
    best_model = f"{OUTPUT_DIR}/checkpoints/best_model.keras"
    if not os.path.exists(best_model):
        raise FileNotFoundError(f"Modèle validé introuvable : {best_model}")
    
    cmd = [
        sys.executable, "-u", "scripts/evaluate_resnet50v2_gtex.py",
        "--config", CONFIG_PATH,
        "--split", "test",
        "--checkpoint", best_model,
        "--dataset-dir", str(DATASET_DIR),
        "--output-dir", OUTPUT_DIR
    ]
    
    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    
    for line in process.stdout:
        print(line, end="")
        
    return_code = process.wait()
    
    if return_code != 0:
        raise RuntimeError("L'évaluation de TEST a échoué.")
        
    if GENERATE_REPORT:
        rep_cmd = [
            sys.executable, "-u", "scripts/generate_resnet50v2_gtex_report.py",
            "--results-dir", OUTPUT_DIR,
            "--class-mapping", str(DATASET_DIR / "metadata" / "class_mapping.json")
        ]
        subprocess.run(rep_cmd, cwd=PROJECT_DIR, env=env, check=True)
else:
    print("RUN_FINAL_TEST = False. Le set de TEST reste scellé.")


In [ ]:
import json
import datetime
import zipfile
import sys
import os
from pathlib import Path

# Generate RUN_MANIFEST.json
manifest = {
    "commit": get_git_commit(),
    "python_version": sys.version,
    "tensorflow_version": tf.__version__ if 'tf' in globals() else "unknown",
    "gpu": gpus[0].name if 'gpus' in globals() and gpus else "None",
    "mixed_precision": tf.keras.mixed_precision.global_policy().name if 'tf' in globals() else "unknown",
    "strategy": tf.distribute.get_strategy().__class__.__name__ if 'tf' in globals() else "unknown",
    "seed": 42,
    "config": CONFIG_PATH,
    "dataset_dir": str(DATASET_DIR) if 'DATASET_DIR' in globals() else "unknown",
    "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "options": {
        "run_tests": RUN_TESTS,
        "run_dataset_audit": RUN_DATASET_AUDIT,
        "run_dry_run": RUN_DRY_RUN,
        "run_smoke_test": RUN_SMOKE_TEST,
        "run_training": RUN_TRAINING,
        "run_validation": RUN_VALIDATION_EVALUATION,
        "run_final_test": RUN_FINAL_TEST,
        "generate_report": GENERATE_REPORT,
        "resume_training": RESUME_TRAINING
    },
    "artifacts_present": {
        "best_model": os.path.exists(f"{OUTPUT_DIR}/checkpoints/best_model.keras"),
        "validation_csv": os.path.exists(f"{OUTPUT_DIR}/validation_predictions.csv"),
        "test_csv": os.path.exists(f"{OUTPUT_DIR}/test_predictions.csv"),
        "dataset_integrity": os.path.exists(f"{OUTPUT_DIR}/dataset_integrity.json")
    },
    "phases_complete": {
        "phase_1": os.path.exists(f"{OUTPUT_DIR}/completion/phase_1.done"),
        "phase_2": os.path.exists(f"{OUTPUT_DIR}/completion/phase_2.done"),
        "phase_3": os.path.exists(f"{OUTPUT_DIR}/completion/phase_3.done")
    },
    "mode": "smoke_test" if RUN_SMOKE_TEST else ("scientific" if RUN_TRAINING else "preflight"),
    "status": "SUCCESS"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
manifest_path = os.path.join(OUTPUT_DIR, "RUN_MANIFEST.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)
print(f"Manifest écrit : {manifest_path}")

# Archive ZIP avec exclusions propres
EXCLUDE_DIRS = {'__pycache__', '.pytest_cache', '.ipynb_checkpoints'}
EXCLUDE_EXTS = {'.pyc', '.pyo'}

output_zip = "/kaggle/working/resnet50v2-gtex-11-exp-a-results.zip"
if os.path.exists(output_zip):
    os.remove(output_zip)

if os.path.exists(OUTPUT_DIR):
    print("Compression des résultats (hors caches)...")
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(OUTPUT_DIR):
            dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
            for file in files:
                if Path(file).suffix in EXCLUDE_EXTS:
                    continue
                filepath = os.path.join(root, file)
                arcname = os.path.relpath(filepath, OUTPUT_DIR)
                zf.write(filepath, arcname)
    print(f"Archive ZIP créée avec succès : {output_zip}")
else:
    print(f"Rien à zipper dans {OUTPUT_DIR}")

print("\nContenu de /kaggle/working :")
for item in Path("/kaggle/working").iterdir():
    print(f" - {item.name}")
